### 후진제거법 : VIF 기반 제거

#### 라이브러리 호출

In [1]:
# 통계 모델링 패키지 설치 
!pip install statsmodels

import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

#### 데이터 불러오기

In [2]:
df = pd.read_csv('single variable_data.csv')

# 초기 20개 변수(스케일링 된 _std 변수) 추출
std_cols = [col for col in df.columns if col.endswith('_std')]
X_scaled = df[std_cols]

print(f"VIF 검사를 진행할 초기 변수 개수: {len(std_cols)}개")

VIF 검사를 진행할 초기 변수 개수: 20개


#### VIF 기반 후진제거법 함수 정의

In [3]:
def backward_elimination_vif(data, threshold=10.0):
    
    # 훼손 방지를 위해 컬럼명 리스트 복사
    features = data.columns.tolist()
    
    print(f"[VIF 기반 후진제거법] 총 {len(features)}개 변수에서 탐색 시작 (임계치: {threshold})\n")
    
    step = 1
    while True:
        # VIF 계산을 위해 상수항(Constant) 추가 (통계적 안정성 확보용)
        X = sm.add_constant(data[features])
        
        # 각 변수별 VIF 계산
        vif_data = pd.DataFrame()
        vif_data["Feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        
        # 상수항(const)은 다중공선성 판단 대상이 아니므로 제외
        vif_data = vif_data[vif_data['Feature'] != 'const']
        
        # VIF 최대값 및 해당 변수 찾기
        max_vif = vif_data['VIF'].max()
        max_feature = vif_data.loc[vif_data['VIF'] == max_vif, 'Feature'].values[0]
        
        # 최대 VIF가 임계치(threshold)보다 크면 해당 변수 제거
        if max_vif > threshold:
            print(f"[Step {step}] 탈락 변수: {max_feature:<25} | VIF: {max_vif:.2f}")
            features.remove(max_feature)
            step += 1
        else:
            print(f"남은 {len(features)}개 모든 변수의 VIF가 {threshold} 이하입니다.")
            break
            
    # 최종 생존 변수 결과 정리 및 출력
    final_vif = vif_data.sort_values(by='VIF', ascending=False).reset_index(drop=True)
    print(f"\n=== [최종 생존 핵심 변수 {len(features)}개 VIF 결과표] ===")
    display(final_vif)
    
    return features

print("VIF 후진제거법 함수 정의 완료!")

VIF 후진제거법 함수 정의 완료!


#### 실행 및 결과 확인

In [4]:
# VIF 임계치 10을 기준으로 후진제거법 실행
surviving_cols_vif = backward_elimination_vif(X_scaled, threshold=10.0)

print("\n[최종 채택된 변수 목록]")
print(surviving_cols_vif)

[VIF 기반 후진제거법] 총 20개 변수에서 탐색 시작 (임계치: 10.0)

남은 20개 모든 변수의 VIF가 10.0 이하입니다.

=== [최종 생존 핵심 변수 20개 VIF 결과표] ===


,Feature,VIF
0,risk_aging_rate_std,3.641075
1,risk_infra_gap_std,3.565838
2,risk_meet_std,2.209312
3,risk_network_std,2.160028
4,risk_nutrition_std,1.820327
5,risk_drug_std,1.792439
6,risk_health_std,1.675877
7,risk_age_std,1.645159
8,risk_depression_std,1.600671
9,risk_digital_std,1.558539



[최종 채택된 변수 목록]
['risk_depression_std', 'risk_drug_std', 'risk_iadl_std', 'risk_health_std', 'risk_age_std', 'risk_housing_tenure_std', 'risk_unemployed_std', 'risk_subj_econ_std', 'risk_housing_safety_std', 'risk_housing_env_std', 'risk_group_std', 'risk_meet_std', 'risk_digital_std', 'risk_nutrition_std', 'risk_smoke_std', 'risk_alcohol_std', 'risk_suicide_std', 'risk_network_std', 'risk_infra_gap_std', 'risk_aging_rate_std']
